# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_01 — GARCH Volatility Modelling

### Research question

How can we model volatility clustering in financial returns using ARCH/GARCH models, estimate parameters by maximum likelihood, compare innovation assumptions, and use conditional volatility forecasts for risk and alpha research?

This notebook opens Phase 3 by moving from **pricing models** into **empirical volatility modelling**.

It connects earlier notebooks:

```text
01_05_empirical_distribution_analyzer
01_10_realized_volatility_estimators
02_06_monte_carlo_gbm_and_fat_tails
02_08_dynamic_delta_hedging_simulation
```

It covers:

1. volatility clustering;
2. ARCH and GARCH intuition;
3. GARCH(1,1) simulation;
4. maximum-likelihood estimation;
5. Gaussian innovations;
6. Student-t innovations;
7. EWMA volatility baseline;
8. conditional volatility forecasts;
9. residual diagnostics;
10. VaR and Expected Shortfall forecasting;
11. VaR backtesting;
12. realised-volatility comparison;
13. volatility-targeting toy signal;
14. limitations and research-frontier extensions.

The key idea:

> GARCH turns volatility from a constant parameter into a time-varying latent state estimated from returns.

## 1. Why GARCH?

Financial returns often show:

- weak linear autocorrelation in raw returns;
- strong autocorrelation in squared returns;
- volatility clustering;
- heavy tails;
- periods of calm followed by periods of stress.

A constant-volatility model assumes:

$$
r_t = \sigma \epsilon_t
$$

where $\sigma$ is fixed.

GARCH instead models conditional variance:

$$
r_t = \mu + \sigma_t \epsilon_t
$$

$$
\sigma_t^2 = \omega + \alpha (r_{t-1}-\mu)^2 + \beta \sigma_{t-1}^2
$$

So volatility changes through time based on:

1. recent shocks;
2. previous volatility.

This makes GARCH a natural first model for forecasting volatility in empirical finance.

## 2. ARCH and GARCH

### ARCH(1)

The ARCH(1) model is:

$$
\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2
$$

where:

$$
\epsilon_t = r_t-\mu
$$

ARCH says volatility reacts to the most recent squared shock.

### GARCH(1,1)

The GARCH(1,1) model adds persistence:

$$
\begin{aligned}
\sigma_t^2 &= \omega \\
&\quad + \alpha \epsilon_{t-1}^2 \\
&\quad + \beta \sigma_{t-1}^2
\end{aligned}
$$

where:

- $\omega>0$;
- $\alpha\geq0$;
- $\beta\geq0$;
- $\alpha+\beta<1$ for covariance stationarity.

The unconditional variance is:

$$
\operatorname{Var}(r_t) = \frac{\omega}{1-\alpha-\beta}
$$

if:

$$
\alpha+\beta<1
$$

The closer $\alpha+\beta$ is to 1, the more persistent volatility is.

## 3. Gaussian vs Student-t innovations

The simplest GARCH model assumes:

$$
\epsilon_t = \sigma_t z_t
$$

$$
z_t\sim\mathcal N(0,1)
$$

But financial returns are often heavier-tailed than Gaussian.

A Student-t GARCH model assumes:

$$
z_t\sim t_\nu
$$

standardised to have variance 1.

This often fits empirical returns better because it allows extreme shocks without forcing volatility alone to explain every tail event.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from math import sqrt
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    from scipy.special import gammaln
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class GARCHConfig:
    """
    GARCH research notebook configuration.
    """
    n_obs: int = 2_500
    annualisation: int = 252
    mu: float = 0.0002
    omega: float = 1.5e-6
    alpha: float = 0.07
    beta: float = 0.91
    student_df: float = 7.0
    seed: int = 42


config = GARCHConfig()
config

## 5. Simulating GARCH(1,1)

We simulate:

$$
r_t = \mu + \sqrt{h_t}z_t
$$

$$
h_t = \omega + \alpha(r_{t-1}-\mu)^2+\beta h_{t-1}
$$

where $h_t=\sigma_t^2$.

For stationarity:

$$
\alpha+\beta<1
$$

and the unconditional variance is:

$$
\frac{\omega}{1-\alpha-\beta}
$$

In [ ]:
def standardised_student_t(rng, df, size):
    """
    Draw Student-t innovations standardised to variance 1.
    """
    raw = rng.standard_t(df=df, size=size)
    return raw * np.sqrt((df - 2.0) / df)


def simulate_garch_11(config: GARCHConfig, innovation: str = "student_t") -> pd.DataFrame:
    """
    Simulate GARCH(1,1) returns.
    """
    rng = np.random.default_rng(config.seed)

    n = config.n_obs
    returns = np.zeros(n)
    variance = np.zeros(n)
    shocks = np.zeros(n)

    unconditional_var = config.omega / (1.0 - config.alpha - config.beta)
    variance[0] = unconditional_var

    if innovation == "gaussian":
        z = rng.standard_normal(n)
    elif innovation == "student_t":
        z = standardised_student_t(rng, config.student_df, n)
    else:
        raise ValueError("innovation must be 'gaussian' or 'student_t'.")

    for t in range(1, n):
        shocks[t - 1] = returns[t - 1] - config.mu
        variance[t] = (
            config.omega
            + config.alpha * shocks[t - 1] ** 2
            + config.beta * variance[t - 1]
        )
        returns[t] = config.mu + np.sqrt(variance[t]) * z[t]

    shocks[-1] = returns[-1] - config.mu

    dates = pd.bdate_range("2015-01-01", periods=n)

    out = pd.DataFrame({
        "date": dates,
        "return": returns,
        "conditional_variance_true": variance,
        "conditional_vol_true": np.sqrt(variance),
        "innovation": z,
        "innovation_type": innovation
    })

    out["price"] = 100 * np.exp(out["return"].cumsum())
    out["squared_return"] = out["return"] ** 2
    out["abs_return"] = out["return"].abs()
    out["annualised_vol_true"] = out["conditional_vol_true"] * np.sqrt(config.annualisation)

    return out


returns_df = simulate_garch_11(config, innovation="student_t")
returns_df.head()

In [ ]:
pd.Series({
    "alpha_plus_beta": config.alpha + config.beta,
    "unconditional_daily_vol": sqrt(config.omega / (1 - config.alpha - config.beta)),
    "unconditional_annualised_vol": sqrt(config.omega / (1 - config.alpha - config.beta)) * sqrt(config.annualisation)
})

## 6. Visualising volatility clustering

Volatility clustering appears as periods of large returns followed by large returns, and calm periods followed by calm periods.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(returns_df["date"], returns_df["price"])
plt.title("Synthetic Price from GARCH Returns")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(returns_df["date"], returns_df["return"])
plt.title("Synthetic Daily Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(returns_df["date"], returns_df["annualised_vol_true"])
plt.title("True Conditional Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.show()

## 7. Autocorrelation diagnostics

GARCH is motivated by the empirical fact that raw returns may have low autocorrelation while squared returns often have persistent autocorrelation.

In [ ]:
def autocorrelation(x, max_lag):
    """
    Simple autocorrelation function.
    """
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.sum(x ** 2)

    rows = []
    for lag in range(1, max_lag + 1):
        num = np.sum(x[lag:] * x[:-lag])
        rows.append({"lag": lag, "autocorrelation": num / denom})

    return pd.DataFrame(rows)


acf_returns = autocorrelation(returns_df["return"], 40).assign(series="returns")
acf_squared = autocorrelation(returns_df["squared_return"], 40).assign(series="squared_returns")
acf_abs = autocorrelation(returns_df["abs_return"], 40).assign(series="absolute_returns")

acf_table = pd.concat([acf_returns, acf_squared, acf_abs], ignore_index=True)
acf_table.head()

In [ ]:
plt.figure(figsize=(10, 6))
for name, group in acf_table.groupby("series"):
    plt.plot(group["lag"], group["autocorrelation"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Autocorrelation of Returns, Squared Returns, and Absolute Returns")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

## 8. Gaussian GARCH likelihood

For Gaussian innovations:

$$
z_t = \frac{r_t-\mu}{\sqrt{h_t}}
$$

$$
z_t\sim\mathcal N(0,1)
$$

The conditional log-likelihood contribution is:

$$
\begin{aligned}
\ell_t &= -\frac12 \Big[ \log(2\pi) \\
&\quad + \log h_t \\
&\quad + \frac{(r_t-\mu)^2}{h_t} \Big]
\end{aligned}
$$

In [ ]:
def transform_garch_params(x):
    """
    Transform unconstrained variables into valid GARCH parameters.
    """
    mu = x[0]
    omega = np.exp(x[1])

    a_raw = np.exp(x[2])
    b_raw = np.exp(x[3])
    denom = 1.0 + a_raw + b_raw

    alpha = a_raw / denom
    beta = b_raw / denom

    return mu, omega, alpha, beta


def compute_garch_variance(returns, mu, omega, alpha, beta):
    """
    Compute GARCH(1,1) conditional variance recursively.
    """
    returns = np.asarray(returns, dtype=float)
    n = len(returns)
    residual = returns - mu

    h = np.zeros(n)

    unconditional = omega / max(1e-8, 1.0 - alpha - beta)
    h[0] = max(unconditional, np.var(returns))

    for t in range(1, n):
        h[t] = omega + alpha * residual[t - 1] ** 2 + beta * h[t - 1]
        h[t] = max(h[t], 1e-12)

    return h


def gaussian_garch_neg_loglik(x, returns):
    mu, omega, alpha, beta = transform_garch_params(x)
    h = compute_garch_variance(returns, mu, omega, alpha, beta)
    residual = returns - mu

    ll = -0.5 * (np.log(2 * np.pi) + np.log(h) + residual ** 2 / h)

    return -float(np.sum(ll))

## 9. Student-t GARCH likelihood

For Student-t innovations, we use standardised Student-t shocks with variance 1.

Student-t GARCH often fits heavy-tailed returns better than Gaussian GARCH.

In [ ]:
def transform_student_garch_params(x):
    """
    Transform unconstrained parameters for Student-t GARCH.
    """
    mu, omega, alpha, beta = transform_garch_params(x[:4])
    df = 2.0 + np.exp(x[4])
    return mu, omega, alpha, beta, df


def standardised_student_logpdf(z, df):
    """
    Log density of Student-t standardised to variance 1.
    """
    scale = np.sqrt((df - 2.0) / df)
    raw = z / scale

    log_pdf_raw = (
        gammaln((df + 1.0) / 2.0)
        - gammaln(df / 2.0)
        - 0.5 * np.log(df * np.pi)
        - ((df + 1.0) / 2.0) * np.log(1.0 + raw ** 2 / df)
    )

    return log_pdf_raw - np.log(scale)


def student_garch_neg_loglik(x, returns):
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for Student-t log-likelihood because gammaln is used.")

    mu, omega, alpha, beta, df = transform_student_garch_params(x)
    h = compute_garch_variance(returns, mu, omega, alpha, beta)
    residual = returns - mu
    z = residual / np.sqrt(h)

    ll = standardised_student_logpdf(z, df) - 0.5 * np.log(h)

    return -float(np.sum(ll))

## 10. Fit Gaussian and Student-t GARCH

If SciPy is available, we estimate both models.

If SciPy is unavailable, the notebook still keeps the simulation and diagnostics runnable, but calibration tables are marked unavailable.

In [ ]:
returns = returns_df["return"].to_numpy()

initial_mu = returns.mean()
initial_var = returns.var()
x0_gaussian = np.array([
    initial_mu,
    np.log(initial_var * 0.02),
    np.log(0.06),
    np.log(0.90)
])

fit_rows = []
fitted_series = {}

if SCIPY_AVAILABLE:
    gaussian_fit = minimize(
        gaussian_garch_neg_loglik,
        x0_gaussian,
        args=(returns,),
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-8, "fatol": 1e-8}
    )

    mu_g, omega_g, alpha_g, beta_g = transform_garch_params(gaussian_fit.x)
    h_g = compute_garch_variance(returns, mu_g, omega_g, alpha_g, beta_g)

    fit_rows.append({
        "model": "gaussian_garch_11",
        "success": bool(gaussian_fit.success),
        "neg_loglik": float(gaussian_fit.fun),
        "mu": mu_g,
        "omega": omega_g,
        "alpha": alpha_g,
        "beta": beta_g,
        "alpha_plus_beta": alpha_g + beta_g,
        "df": np.nan,
        "aic": 2 * 4 + 2 * gaussian_fit.fun,
        "bic": np.log(len(returns)) * 4 + 2 * gaussian_fit.fun
    })

    fitted_series["gaussian"] = h_g

    x0_student = np.array([
        initial_mu,
        np.log(initial_var * 0.02),
        np.log(0.06),
        np.log(0.90),
        np.log(8.0 - 2.0)
    ])

    student_fit = minimize(
        student_garch_neg_loglik,
        x0_student,
        args=(returns,),
        method="Nelder-Mead",
        options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-8}
    )

    mu_t, omega_t, alpha_t, beta_t, df_t = transform_student_garch_params(student_fit.x)
    h_t = compute_garch_variance(returns, mu_t, omega_t, alpha_t, beta_t)

    fit_rows.append({
        "model": "student_t_garch_11",
        "success": bool(student_fit.success),
        "neg_loglik": float(student_fit.fun),
        "mu": mu_t,
        "omega": omega_t,
        "alpha": alpha_t,
        "beta": beta_t,
        "alpha_plus_beta": alpha_t + beta_t,
        "df": df_t,
        "aic": 2 * 5 + 2 * student_fit.fun,
        "bic": np.log(len(returns)) * 5 + 2 * student_fit.fun
    })

    fitted_series["student_t"] = h_t

else:
    fit_rows.append({
        "model": "scipy_unavailable",
        "success": False,
        "neg_loglik": np.nan,
        "mu": np.nan,
        "omega": np.nan,
        "alpha": np.nan,
        "beta": np.nan,
        "alpha_plus_beta": np.nan,
        "df": np.nan,
        "aic": np.nan,
        "bic": np.nan
    })

fit_summary = pd.DataFrame(fit_rows)
fit_summary

## 11. Compare fitted conditional volatility

We compare fitted annualised conditional volatility against the true simulated volatility.

In real data, the true volatility is hidden. Here, because we simulated the data, we can judge fit quality directly.

In [ ]:
vol_fit_df = returns_df[["date", "return", "conditional_vol_true", "annualised_vol_true"]].copy()

if "gaussian" in fitted_series:
    vol_fit_df["gaussian_conditional_vol"] = np.sqrt(fitted_series["gaussian"])
    vol_fit_df["gaussian_annualised_vol"] = vol_fit_df["gaussian_conditional_vol"] * np.sqrt(config.annualisation)

if "student_t" in fitted_series:
    vol_fit_df["student_t_conditional_vol"] = np.sqrt(fitted_series["student_t"])
    vol_fit_df["student_t_annualised_vol"] = vol_fit_df["student_t_conditional_vol"] * np.sqrt(config.annualisation)

vol_fit_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(vol_fit_df["date"], vol_fit_df["annualised_vol_true"], label="True vol", linewidth=2)

if "gaussian_annualised_vol" in vol_fit_df:
    plt.plot(vol_fit_df["date"], vol_fit_df["gaussian_annualised_vol"], label="Gaussian GARCH", alpha=0.8)

if "student_t_annualised_vol" in vol_fit_df:
    plt.plot(vol_fit_df["date"], vol_fit_df["student_t_annualised_vol"], label="Student-t GARCH", alpha=0.8)

plt.title("True vs Fitted Conditional Annualised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

In [ ]:
fit_quality_rows = []

for model_name, col in [
    ("gaussian_garch", "gaussian_annualised_vol"),
    ("student_t_garch", "student_t_annualised_vol")
]:
    if col in vol_fit_df:
        error = vol_fit_df[col] - vol_fit_df["annualised_vol_true"]
        fit_quality_rows.append({
            "model": model_name,
            "vol_rmse": float(np.sqrt(np.mean(error ** 2))),
            "vol_mae": float(np.mean(np.abs(error))),
            "correlation_with_true_vol": float(vol_fit_df[col].corr(vol_fit_df["annualised_vol_true"]))
        })

vol_fit_quality = pd.DataFrame(fit_quality_rows)
vol_fit_quality

## 12. EWMA volatility baseline

RiskMetrics-style EWMA volatility is:

$$
h_t = \lambda h_{t-1} + (1-\lambda)r_{t-1}^2
$$

A common value is:

$$
\lambda=0.94
$$

EWMA is simple and often a strong baseline.

A GARCH model should be compared against simple baselines, not only against itself.

In [ ]:
def ewma_variance(returns, lam=0.94):
    returns = np.asarray(returns, dtype=float)
    h = np.zeros(len(returns))
    h[0] = np.var(returns)

    for t in range(1, len(returns)):
        h[t] = lam * h[t - 1] + (1 - lam) * returns[t - 1] ** 2

    return h


vol_fit_df["ewma_conditional_vol"] = np.sqrt(ewma_variance(returns, lam=0.94))
vol_fit_df["ewma_annualised_vol"] = vol_fit_df["ewma_conditional_vol"] * np.sqrt(config.annualisation)

ewma_error = vol_fit_df["ewma_annualised_vol"] - vol_fit_df["annualised_vol_true"]

ewma_quality = pd.Series({
    "model": "ewma_094",
    "vol_rmse": float(np.sqrt(np.mean(ewma_error ** 2))),
    "vol_mae": float(np.mean(np.abs(ewma_error))),
    "correlation_with_true_vol": float(vol_fit_df["ewma_annualised_vol"].corr(vol_fit_df["annualised_vol_true"]))
})

ewma_quality

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(vol_fit_df["date"], vol_fit_df["annualised_vol_true"], label="True vol", linewidth=2)
plt.plot(vol_fit_df["date"], vol_fit_df["ewma_annualised_vol"], label="EWMA 0.94", alpha=0.85)
if "student_t_annualised_vol" in vol_fit_df:
    plt.plot(vol_fit_df["date"], vol_fit_df["student_t_annualised_vol"], label="Student-t GARCH", alpha=0.85)
plt.title("EWMA vs GARCH Conditional Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 13. Standardised residual diagnostics

A fitted GARCH model should leave residuals that are closer to iid.

Standardised residuals are:

$$
z_t=\frac{r_t-\hat\mu}{\hat\sigma_t}
$$

If squared residual autocorrelation remains, the volatility model is missing structure.

In [ ]:
def residual_diagnostics(returns, mu, variance, model_name):
    resid = returns - mu
    z = resid / np.sqrt(variance)

    mean = float(np.mean(z))
    std = float(np.std(z, ddof=1))
    centered = z - mean
    skew = float(np.mean(centered ** 3) / std ** 3)
    excess_kurtosis = float(np.mean(centered ** 4) / std ** 4 - 3)

    acf_z2_1 = autocorrelation(z ** 2, 1)["autocorrelation"].iloc[0]
    acf_z2_5 = autocorrelation(z ** 2, 5)["autocorrelation"].iloc[-1]

    return {
        "model": model_name,
        "std_resid_mean": mean,
        "std_resid_std": std,
        "std_resid_skew": skew,
        "std_resid_excess_kurtosis": excess_kurtosis,
        "acf_squared_std_resid_lag1": float(acf_z2_1),
        "acf_squared_std_resid_lag5": float(acf_z2_5)
    }, z


resid_rows = []
standardised_residuals = {}

if SCIPY_AVAILABLE:
    g = fit_summary[fit_summary["model"] == "gaussian_garch_11"].iloc[0]
    diag, z = residual_diagnostics(returns, g["mu"], fitted_series["gaussian"], "gaussian_garch")
    resid_rows.append(diag)
    standardised_residuals["gaussian_garch"] = z

    t = fit_summary[fit_summary["model"] == "student_t_garch_11"].iloc[0]
    diag, z = residual_diagnostics(returns, t["mu"], fitted_series["student_t"], "student_t_garch")
    resid_rows.append(diag)
    standardised_residuals["student_t_garch"] = z

residual_report = pd.DataFrame(resid_rows)
residual_report

In [ ]:
if standardised_residuals:
    plt.figure(figsize=(10, 6))
    for name, z in standardised_residuals.items():
        clipped = np.clip(z, np.quantile(z, 0.005), np.quantile(z, 0.995))
        plt.hist(clipped, bins=80, alpha=0.5, density=True, label=name)
    plt.title("Standardised Residual Distribution")
    plt.xlabel("Standardised residual")
    plt.ylabel("Density")
    plt.legend()
    plt.show()

## 14. One-step variance forecasts

For GARCH(1,1), the one-step-ahead variance forecast is:

$$
\begin{aligned}
\hat h_{t+1} &= \omega \\
&\quad + \alpha(r_t-\mu)^2 \\
&\quad + \beta h_t
\end{aligned}
$$

This is useful for risk forecasts, volatility targeting, and alpha features.

In [ ]:
def one_step_variance_forecast(returns, mu, omega, alpha, beta, h):
    residual = returns - mu
    forecast = omega + alpha * residual ** 2 + beta * h
    return forecast


forecast_df = returns_df[["date", "return", "conditional_variance_true"]].copy()

if SCIPY_AVAILABLE:
    for label, model in [("gaussian", "gaussian_garch_11"), ("student_t", "student_t_garch_11")]:
        row = fit_summary[fit_summary["model"] == model].iloc[0]
        h = fitted_series[label]
        forecast_df[f"{label}_variance_forecast_1d"] = one_step_variance_forecast(
            returns,
            row["mu"],
            row["omega"],
            row["alpha"],
            row["beta"],
            h
        )
        forecast_df[f"{label}_annualised_vol_forecast_1d"] = (
            np.sqrt(forecast_df[f"{label}_variance_forecast_1d"]) * np.sqrt(config.annualisation)
        )

forecast_df.head()

In [ ]:
if SCIPY_AVAILABLE:
    plt.figure(figsize=(12, 6))
    plt.plot(forecast_df["date"], np.sqrt(forecast_df["conditional_variance_true"]) * np.sqrt(config.annualisation), label="True next state vol")
    plt.plot(forecast_df["date"], forecast_df["student_t_annualised_vol_forecast_1d"], label="Student-t GARCH forecast", alpha=0.85)
    plt.title("One-Step-Ahead Annualised Volatility Forecast")
    plt.xlabel("Date")
    plt.ylabel("Annualised volatility")
    plt.legend()
    plt.show()

## 15. VaR and Expected Shortfall forecasting

For a one-day return forecast:

$$
r_{t+1}=\mu+\sqrt{h_{t+1}}z_{t+1}
$$

We estimate VaR and Expected Shortfall by drawing from the fitted innovation distribution.

In [ ]:
def empirical_var_es_from_standardised_draws(mu, h, draws, alpha=0.99):
    """
    Compute VaR and ES from standardised innovation draws.
    Loss = -return.
    """
    scenario_returns = mu + np.sqrt(h) * draws
    losses = -scenario_returns
    var = np.quantile(losses, alpha)
    es = losses[losses >= var].mean()
    return var, es


def build_var_forecasts(forecast_df, fit_summary, fitted_series, alpha=0.99, seed=123):
    rng = np.random.default_rng(seed)
    rows = []

    if not SCIPY_AVAILABLE:
        return pd.DataFrame()

    for model_label, model_name in [("gaussian", "gaussian_garch_11"), ("student_t", "student_t_garch_11")]:
        row = fit_summary[fit_summary["model"] == model_name].iloc[0]
        mu = row["mu"]

        if model_label == "gaussian":
            draws = rng.standard_normal(250_000)
        else:
            df = row["df"]
            draws = standardised_student_t(rng, df, 250_000)

        h_forecast = forecast_df[f"{model_label}_variance_forecast_1d"].to_numpy()

        for i, h in enumerate(h_forecast):
            var, es = empirical_var_es_from_standardised_draws(mu, h, draws, alpha=alpha)
            rows.append({
                "date": forecast_df["date"].iloc[i],
                "model": model_label,
                "alpha": alpha,
                "VaR": var,
                "ExpectedShortfall": es
            })

    return pd.DataFrame(rows)


var_forecasts = build_var_forecasts(
    forecast_df,
    fit_summary,
    fitted_series,
    alpha=0.99,
    seed=777
)

var_forecasts.head()

## 16. VaR backtesting

A VaR exceedance occurs when realised loss exceeds forecast VaR:

$$
- r_{t+1} > VaR_t
$$

For a 99% VaR, the expected exceedance rate is about 1%.

In [ ]:
def var_backtest(var_forecasts, returns_df):
    if var_forecasts.empty:
        return pd.DataFrame(), pd.DataFrame()

    realised = returns_df[["date", "return"]].copy()
    realised["next_return"] = realised["return"].shift(-1)
    realised["next_loss"] = -realised["next_return"]

    merged = var_forecasts.merge(realised[["date", "next_loss"]], on="date", how="left").dropna()
    merged["exceedance"] = merged["next_loss"] > merged["VaR"]

    summary = (
        merged
        .groupby(["model", "alpha"])
        .agg(
            n=("exceedance", "count"),
            exceedances=("exceedance", "sum"),
            exceedance_rate=("exceedance", "mean"),
            expected_rate=("alpha", lambda x: 1 - x.iloc[0]),
            mean_VaR=("VaR", "mean"),
            mean_ES=("ExpectedShortfall", "mean")
        )
        .reset_index()
    )

    return merged, summary


var_backtest_detail, var_backtest_summary = var_backtest(var_forecasts, returns_df)
var_backtest_summary

In [ ]:
if not var_backtest_detail.empty:
    plt.figure(figsize=(12, 6))
    for model, group in var_backtest_detail.groupby("model"):
        plt.plot(group["date"], group["VaR"], label=f"{model} VaR", alpha=0.85)
    plt.plot(var_backtest_detail["date"].unique(), var_backtest_detail.groupby("date")["next_loss"].first(), label="Next-day realised loss", alpha=0.45)
    plt.title("One-Day 99% VaR Forecasts")
    plt.xlabel("Date")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

## 17. Realised volatility comparison

From Phase 1, realised volatility estimators gave daily variance features.

Here we create a simple rolling realised variance proxy:

$$
RV_t = \sum_{i=t-20}^{t} r_i^2
$$

annualised as:

$$
\sqrt{252 \cdot RV_t / 21}
$$

In [ ]:
rv_window = 21

vol_fit_df["rolling_realised_variance_21d"] = returns_df["return"].rolling(rv_window).apply(lambda x: np.sum(x**2), raw=True)
vol_fit_df["rolling_realised_vol_21d_ann"] = np.sqrt(
    config.annualisation * vol_fit_df["rolling_realised_variance_21d"] / rv_window
)

plt.figure(figsize=(12, 6))
plt.plot(vol_fit_df["date"], vol_fit_df["rolling_realised_vol_21d_ann"], label="Rolling realised vol 21d", alpha=0.8)
if "student_t_annualised_vol" in vol_fit_df:
    plt.plot(vol_fit_df["date"], vol_fit_df["student_t_annualised_vol"], label="Student-t GARCH conditional vol", alpha=0.8)
plt.title("Rolling Realised Volatility vs GARCH Conditional Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 18. Volatility-targeting toy signal

A simple use of volatility forecasts is volatility targeting.

Position size:

$$
w_t = \frac{\sigma_{\text{target}}}{\hat\sigma_t}
$$

with a leverage cap:

$$
w_t \leq w_{\max}
$$

This is not an alpha strategy by itself. It is a risk-scaling mechanism.

In [ ]:
def volatility_targeting_backtest(returns, vol_forecast_ann, target_vol=0.15, max_leverage=2.0):
    """
    Simple volatility-targeted return stream.
    """
    vol = np.asarray(vol_forecast_ann, dtype=float)
    weights = target_vol / np.maximum(vol, 1e-6)
    weights = np.clip(weights, 0.0, max_leverage)

    strategy_returns = weights[:-1] * returns[1:]

    return pd.DataFrame({
        "weight": weights[:-1],
        "next_return": returns[1:],
        "strategy_return": strategy_returns
    })


if "student_t_annualised_vol_forecast_1d" in forecast_df:
    vt = volatility_targeting_backtest(
        returns=forecast_df["return"].to_numpy(),
        vol_forecast_ann=forecast_df["student_t_annualised_vol_forecast_1d"].to_numpy(),
        target_vol=0.15,
        max_leverage=2.0
    )

    vt["cum_strategy"] = (1 + vt["strategy_return"]).cumprod()
    vt["cum_underlying"] = (1 + vt["next_return"]).cumprod()

    display(vt.head())
else:
    vt = pd.DataFrame()

In [ ]:
if not vt.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(vt["cum_strategy"], label="Vol-targeted")
    plt.plot(vt["cum_underlying"], label="Underlying")
    plt.title("Toy Volatility-Targeting Backtest")
    plt.xlabel("Time")
    plt.ylabel("Cumulative growth")
    plt.legend()
    plt.show()

    vt_summary = pd.Series({
        "strategy_ann_vol": vt["strategy_return"].std() * np.sqrt(config.annualisation),
        "underlying_ann_vol": vt["next_return"].std() * np.sqrt(config.annualisation),
        "strategy_mean_daily": vt["strategy_return"].mean(),
        "underlying_mean_daily": vt["next_return"].mean(),
        "mean_weight": vt["weight"].mean(),
        "max_weight": vt["weight"].max()
    })
else:
    vt_summary = pd.Series(dtype=float)

vt_summary

## 19. Practical checklist for GARCH modelling

Before trusting a GARCH model, check:

1. **Return scaling** — are returns decimals or percentages?
2. **Stationarity** — is $\alpha+\beta<1$?
3. **Innovation distribution** — Gaussian or Student-t?
4. **Residual diagnostics** — are standardised residuals close to iid?
5. **Squared residual autocorrelation** — does volatility clustering remain?
6. **Forecast evaluation** — are volatility forecasts tested out of sample?
7. **Risk backtesting** — does VaR exceedance rate match expectation?
8. **Baseline comparison** — does GARCH beat EWMA or realised-vol baselines?
9. **Parameter stability** — are parameters stable through time?
10. **Use case** — is the model for risk, pricing, sizing, or alpha features?

## 20. Saving outputs

The notebook saves:

1. simulated GARCH returns;
2. autocorrelation diagnostics;
3. fit summary;
4. volatility fit comparison;
5. residual diagnostics;
6. one-step forecasts;
7. VaR forecasts and backtests;
8. realised-vol comparison;
9. volatility-targeting toy results;
10. manifest.

In [ ]:
output_dir = Path("data/processed/garch_volatility_modeling_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "simulated_garch_returns.csv"
acf_path = output_dir / "autocorrelation_diagnostics.csv"
fit_summary_path = output_dir / "garch_fit_summary.csv"
vol_fit_path = output_dir / "conditional_volatility_fit.csv"
vol_fit_quality_path = output_dir / "volatility_fit_quality.csv"
residual_report_path = output_dir / "standardised_residual_diagnostics.csv"
forecast_path = output_dir / "one_step_variance_forecasts.csv"
var_forecasts_path = output_dir / "var_es_forecasts.csv"
var_backtest_path = output_dir / "var_backtest_summary.csv"
vt_path = output_dir / "volatility_targeting_toy_backtest.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
acf_table.to_csv(acf_path, index=False)
fit_summary.to_csv(fit_summary_path, index=False)
vol_fit_df.to_csv(vol_fit_path, index=False)

all_quality = vol_fit_quality.copy()
if not ewma_quality.empty:
    all_quality = pd.concat([all_quality, ewma_quality.to_frame().T], ignore_index=True)
all_quality.to_csv(vol_fit_quality_path, index=False)

residual_report.to_csv(residual_report_path, index=False)
forecast_df.to_csv(forecast_path, index=False)
var_forecasts.to_csv(var_forecasts_path, index=False)
var_backtest_summary.to_csv(var_backtest_path, index=False)
vt.to_csv(vt_path, index=False)

manifest = {
    "dataset_name": "garch_volatility_modeling_outputs",
    "schema_version": "garch_volatility_modeling_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "scipy_available": SCIPY_AVAILABLE,
    "fit_summary": fit_summary.to_dict(orient="records"),
    "volatility_targeting_summary": vt_summary.to_dict(),
    "notes": [
        "Synthetic returns are generated from Student-t GARCH(1,1).",
        "Gaussian and Student-t GARCH are estimated by maximum likelihood when SciPy is available.",
        "EWMA volatility is included as a baseline.",
        "VaR and ES forecasts use simulated innovation quantiles.",
        "Volatility-targeting section is a toy risk-scaling demonstration, not an alpha claim."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, fit_summary_path, vol_fit_path, var_backtest_path, manifest_path

## 21. Limitations

### 21.1 Synthetic data

The notebook uses synthetic returns with known true volatility.

Real data introduces structural breaks, market microstructure effects, missing values, and changing regimes.

### 21.2 GARCH(1,1) only

The baseline model is GARCH(1,1).

Extensions may fit volatility asymmetry, leverage effects, long memory, or realised-volatility inputs.

### 21.3 Constant parameters

The model assumes fixed parameters.

Real markets can have time-varying parameters and regime changes.

### 21.4 Innovation assumptions

Gaussian and Student-t innovations are still stylised.

Real return tails may be skewed, jumpy, or regime-dependent.

### 21.5 MLE sensitivity

Maximum-likelihood estimation can depend on initial guesses, scaling, optimiser choice, and constraints.

### 21.6 VaR backtesting is simple

The notebook uses simple exceedance rates.

Production risk systems use more formal unconditional and conditional coverage tests.

### 21.7 Vol targeting is not alpha

Volatility targeting changes risk exposure.

It does not create alpha by itself unless combined with forecasts that improve risk-adjusted returns or drawdown control.

## 22. Research frontier and extensions

Important extensions include:

1. **EGARCH** — models log variance and asymmetry.
2. **GJR-GARCH / threshold GARCH** — captures leverage effects where negative returns increase volatility more than positive returns.
3. **Realised GARCH** — incorporates realised volatility measures from intraday data.
4. **HAR-RV** — uses daily, weekly, and monthly realised volatility components.
5. **GARCH with skewed-t innovations** — captures asymmetric heavy tails.
6. **Regime-switching GARCH** — allows parameters to change across market regimes.
7. **Multivariate GARCH** — models covariance dynamics across assets.
8. **Neural volatility models** — uses deep learning for nonlinear volatility forecasting.
9. **Option-implied volatility hybrids** — combines GARCH forecasts with implied volatility surfaces.
10. **Chinese futures volatility modelling** — includes night sessions, contract rolls, price limits, and product-specific seasonality.

## 23. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_02_har_realized_volatility_forecasting**  
   Forecasting realised volatility using daily, weekly, and monthly components.

2. **03_03_regime_detection_hidden_markov_models**  
   Identifying volatility regimes.

3. **03_04_factor_momentum_and_volatility_scaling**  
   Using volatility forecasts in alpha/risk scaling.

4. **03_05_cross_sectional_alpha_features**  
   Turning volatility into cross-sectional features.

5. **04_06_var_cvar_and_stress_testing**  
   Using GARCH forecasts in portfolio risk.

6. **05_01_vectorized_backtest_engine**  
   Testing volatility-targeted strategies.

7. **07_01_multi_asset_cta_research_pipeline**  
   Applying volatility forecasting to futures trend strategies.

## 24. Summary

This notebook implemented GARCH volatility modelling.

It showed how to:

1. simulate Student-t GARCH returns;
2. visualise volatility clustering;
3. diagnose autocorrelation in squared returns;
4. estimate Gaussian GARCH by maximum likelihood;
5. estimate Student-t GARCH by maximum likelihood;
6. compare fitted conditional volatility against true volatility;
7. benchmark against EWMA volatility;
8. analyse standardised residuals;
9. produce one-step volatility forecasts;
10. build VaR and Expected Shortfall forecasts;
11. backtest VaR exceedances;
12. compare GARCH volatility to rolling realised volatility;
13. use forecasts in a toy volatility-targeting rule.

The key computational takeaway:

> GARCH is a recursive latent-volatility filter estimated by likelihood, and its diagnostics matter as much as its fitted line.

The key financial takeaway:

> Volatility forecasts are research infrastructure. They support risk sizing, VaR, hedging diagnostics, and alpha feature engineering.

## 25. Further reading

- Engle, R. "Autoregressive Conditional Heteroskedasticity with Estimates of the Variance of United Kingdom Inflation."
- Bollerslev, T. "Generalized Autoregressive Conditional Heteroskedasticity."
- Taylor, S. *Modelling Financial Time Series.*
- Tsay, R. *Analysis of Financial Time Series.*
- McNeil, Frey, and Embrechts, *Quantitative Risk Management.*
- Literature on EGARCH, GJR-GARCH, realised GARCH, HAR-RV, and multivariate GARCH.